In [1]:
import os
import tensorflow as tf
from tensorflow.keras.layers import Layer, GlobalAveragePooling2D, Dense, Reshape, Multiply
import joblib
import pandas as pd

class ChannelAttention(Layer):
    """Custom Channel Attention Module"""
    def __init__(self, ratio=8, **kwargs):
        super(ChannelAttention, self).__init__(**kwargs)
        self.ratio = ratio

    def build(self, input_shape):
        channels = input_shape[-1]
        self.gap = GlobalAveragePooling2D()
        self.dense1 = Dense(max(1, channels // self.ratio), activation='relu')
        self.dense2 = Dense(channels, activation='sigmoid')
        self.reshape = Reshape((1, 1, channels))
        super().build(input_shape)

    def call(self, x):
        attn = self.gap(x)
        attn = self.dense1(attn)
        attn = self.dense2(attn)
        # Fix missing dimension issue if needed, reshape directly creates 1x1xC shape
        return Multiply()([x, attn])

    def get_config(self):
        config = super().get_config()
        config.update({'ratio': self.ratio})
        return config

def format_size(size_bytes):
    if size_bytes < 1024:
        return f"{size_bytes} B"
    elif size_bytes < 1024 * 1024:
        return f"{size_bytes / 1024:.2f} KB"
    else:
        return f"{size_bytes / (1024 * 1024):.2f} MB"

model_dir = "saved_models_20260427_133743"
loaded_models = {}

print(f"Loading models from {model_dir}...\n")

main_models = ["feature_extractor.keras", "svm_classifier.pkl"]
model_info = []
additional_files_info = []
total_bytes = 0

for filename in os.listdir(model_dir):
    filepath = os.path.join(model_dir, filename)
    if not os.path.isfile(filepath):
        continue
        
    size_bytes = os.path.getsize(filepath)
    size_str = format_size(size_bytes)
    
    status = "Loaded"
    try:
        if filename.endswith(".keras") or filename.endswith(".h5"):
            loaded_models[filename] = tf.keras.models.load_model(
                filepath, 
                custom_objects={'ChannelAttention': ChannelAttention}
            )
        elif filename.endswith(".pkl"):
            loaded_models[filename] = joblib.load(filepath)
        else:
            status = "Ignored"  # File JSON atau TXT tidak perlu diload sebagai model
    except Exception as e:
        status = f"Error"

    if filename in main_models:
        model_info.append({
            "Nama File": filename,
            "Ukuran": size_str,
            "Ukuran (Bytes)": size_bytes,
            "Load Status": status
        })
        total_bytes += size_bytes
    else:
        desc = "untuk preprocessing" if "scaler" in filename else "dokumentasi training" if "history" in filename else "informasi model"
        # Tambahkan status load jika itu file .pkl
        load_tag = f" [Status: {status}]" if filename.endswith(".pkl") else ""
        additional_files_info.append(f"  • {filename} ({size_str}) - {desc}{load_tag}")

print("TABEL UKURAN MODEL SAAT DI-LOAD:")
print(f"{'Nama File':>25} {'Ukuran':>10} {'Ukuran (Bytes)':>15} {'Load Status':>15}")
for item in model_info:
    print(f"{item['Nama File']:>25} {item['Ukuran']:>10} {item['Ukuran (Bytes)']:>15} {item['Load Status']:>15}")
print(f"{'═══ TOTAL MODEL ═══':>25} {format_size(total_bytes):>10} {total_bytes:>15}")

print("\n================================================================================")
print("Catatan: Ukuran di atas hanya mencakup file model produksi.")
print("File tambahan yang juga tersimpan:")
for info in sorted(additional_files_info):
    print(info)

Loading models from saved_models_20260427_133743...

TABEL UKURAN MODEL SAAT DI-LOAD:
                Nama File     Ukuran  Ukuran (Bytes)     Load Status
  feature_extractor.keras   12.44 MB        13044760          Loaded
       svm_classifier.pkl    8.94 KB            9155          Loaded
      ═══ TOTAL MODEL ═══   12.45 MB        13053915

Catatan: Ukuran di atas hanya mencakup file model produksi.
File tambahan yang juga tersimpan:
  • cnn_attention_model.keras (37.29 MB) - informasi model
  • feature_scaler.pkl (999 B) - untuk preprocessing [Status: Loaded]
  • model_info.txt (989 B) - informasi model
  • training_history.json (1.56 KB) - dokumentasi training


In [30]:
def build_size_table(artifact_paths, svm_orig_path, scaler_orig_path, svm_lowrank_path, scaler_lowrank_path):
    # Gunakan folder saved_models_20260427_133743 sebagai acuan model asli (Awal)
    model_dir = "saved_models_20260427_133743"
    
    rows = [
        {
            'Komponen': 'CNN Attention (main)',
            'Awal (KB)': _file_size_kb(os.path.join(model_dir, 'cnn_attention_model.keras')),
            'Setelah Low-Rank (KB)': _file_size_kb(artifact_paths['cnn_lowrank']),
            'Setelah Low-Rank + GZIP (KB)': _file_size_kb(artifact_paths['cnn_lowrank_gz']),
        },
        {
            'Komponen': 'CNN Feature Extractor',
            'Awal (KB)': _file_size_kb(os.path.join(model_dir, 'feature_extractor.keras')),
            'Setelah Low-Rank (KB)': _file_size_kb(artifact_paths['extractor_lowrank']),
            'Setelah Low-Rank + GZIP (KB)': _file_size_kb(artifact_paths['extractor_lowrank_gz']),
        },
        {
            'Komponen': 'SVM Classifier',
            'Awal (KB)': _file_size_kb(os.path.join(model_dir, 'svm_classifier.pkl')),
            'Setelah Low-Rank (KB)': _file_size_kb(svm_lowrank_path),
            'Setelah Low-Rank + GZIP (KB)': _file_size_kb(svm_lowrank_path),
        },
        {
            'Komponen': 'Feature Scaler',
            'Awal (KB)': _file_size_kb(os.path.join(model_dir, 'feature_scaler.pkl')),
            'Setelah Low-Rank (KB)': _file_size_kb(scaler_lowrank_path),
            'Setelah Low-Rank + GZIP (KB)': _file_size_kb(scaler_lowrank_path),
        },
    ]
    df = pd.DataFrame(rows)
    total_row = {
        'Komponen': 'TOTAL',
        'Awal (KB)': df['Awal (KB)'].sum(),
        'Setelah Low-Rank (KB)': df['Setelah Low-Rank (KB)'].sum(),
        'Setelah Low-Rank + GZIP (KB)': df['Setelah Low-Rank + GZIP (KB)'].sum(),
    }
    return pd.concat([df, pd.DataFrame([total_row])], ignore_index=True)


def run_complete_pipeline(model, X_train, X_valid, X_test, y_train, y_valid, y_test):
    y_train_int = np.argmax(y_train, axis=1)
    y_valid_int = np.argmax(y_valid, axis=1)
    y_test_int = np.argmax(y_test, axis=1)

    artifact_paths, lowrank_model, extractor_lowrank, history_lowrank = setup_lowrank_pipeline(
        model, X_train, X_valid, y_train, y_valid
    )

    print("\n" + "=" * 70)
    print("PIPELINE A: ORIGINAL MODEL (float32)")
    print("=" * 70)

    # Menggunakan model original dari file asli untuk perhitungan metric ukuran
    model_dir = "saved_models_20260427_133743"
    size_orig_kb = _file_size_kb(os.path.join(model_dir, 'cnn_attention_model.keras'))
    
    extractor_orig = Model(inputs=model.input, outputs=model.get_layer('feature_layer').output)

    t0 = time.perf_counter()
    X_train_feat_orig = extract_features_keras(extractor_orig, X_train)
    X_val_feat_orig = extract_features_keras(extractor_orig, X_valid)
    X_test_feat_orig = extract_features_keras(extractor_orig, X_test)
    t_feat_orig = time.perf_counter() - t0

    scaler_orig = StandardScaler()
    X_train_scaled_orig = scaler_orig.fit_transform(X_train_feat_orig)
    X_val_scaled_orig = scaler_orig.transform(X_val_feat_orig)
    X_test_scaled_orig = scaler_orig.transform(X_test_feat_orig)

    X_svm_train_orig = np.vstack([X_train_scaled_orig, X_val_scaled_orig])
    y_svm_train_orig = np.concatenate([y_train_int, y_valid_int])

    t0 = time.perf_counter()
    svm_orig = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
    svm_orig.fit(X_svm_train_orig, y_svm_train_orig)
    t_svm_orig = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_pred_orig = svm_orig.predict(X_test_scaled_orig)
    y_proba_orig = svm_orig.predict_proba(X_test_scaled_orig)
    t_inf_orig = time.perf_counter() - t0

    acc_orig = accuracy_score(y_test_int, y_pred_orig)

    print("\n" + "=" * 70)
    print("PIPELINE B: LOW-RANK MODEL")
    print("=" * 70)

    size_lowrank_kb = _file_size_kb(artifact_paths['cnn_lowrank'])
    size_lowrank_gz_kb = _file_size_kb(artifact_paths['cnn_lowrank_gz'])

    t0 = time.perf_counter()
    X_train_feat_lowrank = extract_features_keras(extractor_lowrank, X_train)
    X_val_feat_lowrank = extract_features_keras(extractor_lowrank, X_valid)
    X_test_feat_lowrank = extract_features_keras(extractor_lowrank, X_test)
    t_feat_lowrank = time.perf_counter() - t0

    scaler_lowrank = StandardScaler()
    X_train_scaled_lowrank = scaler_lowrank.fit_transform(X_train_feat_lowrank)
    X_val_scaled_lowrank = scaler_lowrank.transform(X_val_feat_lowrank)
    X_test_scaled_lowrank = scaler_lowrank.transform(X_test_feat_lowrank)

    X_svm_train_lowrank = np.vstack([X_train_scaled_lowrank, X_val_scaled_lowrank])
    y_svm_train_lowrank = np.concatenate([y_train_int, y_valid_int])

    t0 = time.perf_counter()
    svm_lowrank = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
    svm_lowrank.fit(X_svm_train_lowrank, y_svm_train_lowrank)
    t_svm_lowrank = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_pred_lowrank = svm_lowrank.predict(X_test_scaled_lowrank)
    y_proba_lowrank = svm_lowrank.predict_proba(X_test_scaled_lowrank)
    t_inf_lowrank = time.perf_counter() - t0

    acc_lowrank = accuracy_score(y_test_int, y_pred_lowrank)
    
    # OUTPUT UNTUK PIPELINE B
    print(f"Model size (lowrank .keras): {size_lowrank_kb:.1f} KB")
    print(f"Model size (lowrank .keras.gz): {size_lowrank_gz_kb:.1f} KB")
    print(f"Feature extraction time: {t_feat_lowrank:.3f} s")
    print(f"SVM training time: {t_svm_lowrank:.3f} s")
    print(f"Inference time: {t_inf_lowrank:.4f} s")
    print(f"Test accuracy: {acc_lowrank:.4f}")

    os.makedirs('artifacts', exist_ok=True)
    svm_orig_path = 'artifacts/svm_original.pkl'
    scaler_orig_path = 'artifacts/scaler_original.pkl'
    svm_lowrank_path = 'artifacts/svm_lowrank.pkl'
    scaler_lowrank_path = 'artifacts/scaler_lowrank.pkl'

    joblib.dump(svm_orig, svm_orig_path)
    joblib.dump(scaler_orig, scaler_orig_path)
    joblib.dump(svm_lowrank, svm_lowrank_path)
    joblib.dump(scaler_lowrank, scaler_lowrank_path)

    size_df = build_size_table(
        artifact_paths,
        svm_orig_path,
        scaler_orig_path,
        svm_lowrank_path,
        scaler_lowrank_path,
    )

    # CLASSIFICATION REPORT UNTUK PIPELINE B
    print("\nClassification Report (Low-Rank):")
    print(classification_report(y_test_int, y_pred_lowrank, zero_division=0))
    print_size_table(size_df)

    metrics = {
        'size_kb': [size_orig_kb, size_lowrank_gz_kb],
        'size_lowrank_raw_kb': [size_orig_kb, size_lowrank_kb],
        't_feature': [t_feat_orig, t_feat_lowrank],
        't_svm': [t_svm_orig, t_svm_lowrank],
        't_infer': [t_inf_orig, t_inf_lowrank],
        'accuracy': [acc_orig, acc_lowrank],
    }

    eval_data = (y_test_int, y_pred_orig, y_pred_lowrank, y_proba_orig, y_proba_lowrank)
    globals()['eval_data'] = eval_data
    globals()['metrics'] = metrics

    return metrics, eval_data, size_df, history_lowrank


In [32]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.utils import to_categorical

dataset_base_path = './dataset_processed2'
categories = ['Bengin cases', 'Malignant cases', 'Normal cases']

def load_split_data(split_folder_path, categories):
    """Load semua gambar dari folder split (train/valid/test)"""
    X = []
    y = []
    
    for class_idx, category in enumerate(categories):
        category_path = os.path.join(split_folder_path, category)
        if not os.path.isdir(category_path):
            continue
        
        image_files = sorted([f for f in os.listdir(category_path) if f.endswith('.jpg')])
        for img_file in image_files:
            img_path = os.path.join(category_path, img_file)
            try:
                img = cv2.imread(img_path)
                if img is None:
                    continue
                img = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (224, 224))
                img = img.astype(np.float32) / 255.0
                X.append(img)
                y.append(class_idx)
            except Exception:
                continue
    
    return np.array(X, dtype=np.float32), np.array(y)

print("Loading dataset...")
X_train, y_train_labels = load_split_data(os.path.join(dataset_base_path, 'train'), categories)
X_valid, y_valid_labels = load_split_data(os.path.join(dataset_base_path, 'valid'), categories)
X_test, y_test_labels = load_split_data(os.path.join(dataset_base_path, 'test'), categories)

num_classes = len(categories)
y_train = to_categorical(y_train_labels, num_classes=num_classes)
y_valid = to_categorical(y_valid_labels, num_classes=num_classes)
y_test = to_categorical(y_test_labels, num_classes=num_classes)

print(f"Dataset loaded: Train ({len(X_train)}), Valid ({len(X_valid)}), Test ({len(X_test)})")

# Gunakan model dari Cell 1 (recommended) - DIPERBAIKI (Menggunakan cnn_attention_model.keras)
if 'loaded_models' in globals() and 'cnn_attention_model.keras' in loaded_models:
    model_to_lowrank = loaded_models['cnn_attention_model.keras']
else:
    if 'ChannelAttention' not in globals():
        raise NameError(
            "ChannelAttention belum tersedia. Jalankan Cell 1 terlebih dahulu agar class custom terdefinisi."
        )
    model_path = os.path.join('saved_models_20260427_133743', 'cnn_attention_model.keras')
    model_to_lowrank = tf.keras.models.load_model(
        model_path,
        custom_objects={'ChannelAttention': ChannelAttention}
    )

print("\nMemulai pipeline low-rank...")
metrics, eval_data, size_df, history_lowrank = run_complete_pipeline(
    model=model_to_lowrank,
    X_train=X_train,
    X_valid=X_valid,
    X_test=X_test,
    y_train=y_train,
    y_valid=y_valid,
    y_test=y_test
)


Loading dataset...
Dataset loaded: Train (737), Valid (158), Test (159)

Memulai pipeline low-rank...
[Low-Rank] Eligible tensors transformed: 8
[Low-Rank] Average retained energy: 0.7284
[Low-Rank] Average target rank: 9.00
Epoch 1/5
24/24 [==============================] - 3s 102ms/step - loss: 0.0286 - accuracy: 0.9946 - val_loss: 0.0417 - val_accuracy: 0.9937
Epoch 2/5
24/24 [==============================] - 2s 96ms/step - loss: 0.0101 - accuracy: 0.9986 - val_loss: 0.0180 - val_accuracy: 0.9937
Epoch 3/5
24/24 [==============================] - 2s 96ms/step - loss: 0.0053 - accuracy: 0.9986 - val_loss: 0.0218 - val_accuracy: 0.9873
Epoch 4/5
24/24 [==============================] - 2s 96ms/step - loss: 0.0077 - accuracy: 0.9986 - val_loss: 0.0350 - val_accuracy: 0.9937

PIPELINE A: ORIGINAL MODEL (float32)

PIPELINE B: LOW-RANK MODEL
Model size (lowrank .keras): 38187.7 KB
Model size (lowrank .keras.gz): 18423.0 KB
Feature extraction time: 1.225 s
SVM training time: 0.008 s
Infer

In [36]:
import os
import shutil
import json
from datetime import datetime

# ===========================================================================
# SAVE ALL LOW-RANK MODELS TO NEW FOLDER
# ===========================================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_folder = f"saved_models_lowrank_{timestamp}"
os.makedirs(save_folder, exist_ok=True)

print(f"=" * 70)
print(f"MENYIMPAN SEMUA KOMPONEN MODEL LOW-RANK KE FOLDER: {save_folder}")
print(f"=" * 70)

artifacts_dir = "artifacts"

# Pemetaan nama file dari output pipeline low-rank -> nama output final
files_mapping = {
    "cnn_attention_lowrank.keras": "cnn_attention_model_lowrank.keras",
    "cnn_attention_extractor_lowrank.keras": "feature_extractor_lowrank.keras"
}

try:
    for src_name, dst_name in files_mapping.items():
        src_path = os.path.join(artifacts_dir, src_name)
        dst_path = os.path.join(save_folder, dst_name)
        
        if os.path.exists(src_path):
            shutil.copy2(src_path, dst_path)
            print(f"Saved: {dst_name} \t(Size: {os.path.getsize(dst_path) / 1024:.2f} KB)")
        else:
            print(f"Warning: File {src_name} tidak ditemukan di folder {artifacts_dir}/")
            
    # Simpan history low-rank agar konsisten dengan output training
    if 'history_lowrank' in globals() and history_lowrank is not None:
        history_dict = {
            'loss': [float(x) for x in history_lowrank.history.get('loss', [])],
            'val_loss': [float(x) for x in history_lowrank.history.get('val_loss', [])],
            'accuracy': [float(x) for x in history_lowrank.history.get('accuracy', [])],
            'val_accuracy': [float(x) for x in history_lowrank.history.get('val_accuracy', [])]
        }
        history_path = os.path.join(save_folder, "training_history_lowrank.json")
        with open(history_path, 'w') as f:
            json.dump(history_dict, f, indent=2)
        print("Saved: training_history_lowrank.json")
        
    # Simpan ringkasan model info
    info_path = os.path.join(save_folder, "model_info_lowrank.txt")
    with open(info_path, 'w') as f:
        f.write("=" * 70 + "\n")
        f.write("LOW-RANK CNN + CHANNEL ATTENTION + SVM MODEL INFORMATION\n")
        f.write("=" * 70 + "\n\n")
        f.write("Model Components:\n")
        f.write("1. Low-Rank CNN Attention Model: cnn_attention_model_lowrank.keras\n")
        f.write("2. Low-Rank CNN Feature Extractor: feature_extractor_lowrank.keras\n")
        f.write("3. SVM Classifier: dibuat saat run_complete_pipeline (in-memory)\n")
        f.write("4. Feature Scaler: dibuat saat run_complete_pipeline (in-memory)\n\n")
        f.write("Keterangan: File disimpan dalam format '.keras' yang direkomendasikan TensorFlow\n")
        f.write("dengan akhiran '_lowrank' karena bobot model telah dikompresi low-rank.\n")
    print("Saved: model_info_lowrank.txt")
            
    print(f"\n{'=' * 80}")
    print(f"SEMUA MODEL LOW-RANK BERHASIL DISIMPAN DI FOLDER {save_folder}")
    print(f"{'=' * 80}")
    
except Exception as e:
    print(f"\nError saat menyalin/menyimpan model: {str(e)}")

MENYIMPAN SEMUA KOMPONEN MODEL LOW-RANK KE FOLDER: saved_models_lowrank_20260427_143812
Saved: cnn_attention_model_lowrank.keras 	(Size: 38187.68 KB)
Saved: feature_extractor_lowrank.keras 	(Size: 12739.02 KB)
Saved: training_history_lowrank.json
Saved: model_info_lowrank.txt

SEMUA MODEL LOW-RANK BERHASIL DISIMPAN DI FOLDER saved_models_lowrank_20260427_143812


In [27]:
import hashlib
import os
from collections import defaultdict

print("═" * 70)
print("MEMERIKSA DUPLIKASI GAMBAR DALAM DATASET")
print("═" * 70)

def get_image_hash(filepath):
    """Menghitung SHA-256 hash dari file gambar."""
    hasher = hashlib.sha256()
    with open(filepath, 'rb') as f:
        buf = f.read()
        hasher.update(buf)
    return hasher.hexdigest()

dataset_path = dataset_base_path
hash_dict = defaultdict(list)
total_files = 0

# Scan semua gambar di dataset
for root, _, files in os.walk(dataset_path):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            filepath = os.path.join(root, file)
            file_hash = get_image_hash(filepath)
            
            # Simpan path relatif terhadap dataset_base_path
            rel_path = os.path.relpath(filepath, dataset_path)
            hash_dict[file_hash].append(rel_path)
            total_files += 1

# Cari duplikat
duplicates = {h: paths for h, paths in hash_dict.items() if len(paths) > 1}

print(f"Total gambar diperiksa: {total_files}")
print(f"Total gambar unik: {len(hash_dict)}")

if duplicates:
    print(f"\n⚠️ DITEMUKAN {len(duplicates)} GAMBAR DUPLIKAT TERSEBAR DI BEBERAPA FILE:")
    for i, (h, paths) in enumerate(duplicates.items(), 1):
        print(f"\nGroup {i} (Hash: {h[:8]}...):")
        for p in paths:
            print(f"  - {p}")
else:
    print("\n✅ TIDAK ADA GAMBAR DUPLIKAT DITEMUKAN DALAM DATASET.")


══════════════════════════════════════════════════════════════════════
MEMERIKSA DUPLIKASI GAMBAR DALAM DATASET
══════════════════════════════════════════════════════════════════════
Total gambar diperiksa: 1054
Total gambar unik: 1054

✅ TIDAK ADA GAMBAR DUPLIKAT DITEMUKAN DALAM DATASET.
